<a href="https://colab.research.google.com/github/zakieassad/pcb-defect-detection-yolo/blob/main/notebooks/04_colab_final_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch: 2.11.0+cu128
CUDA disponible: True
GPU: Tesla T4


In [2]:
!git clone https://github.com/zakieassad/pcb-defect-detection-yolo.git
%cd pcb-defect-detection-yolo

Cloning into 'pcb-defect-detection-yolo'...
remote: Enumerating objects: 35, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 35 (delta 12), reused 25 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (35/35), 665.03 KiB | 2.76 MiB/s, done.
Resolving deltas: 100% (12/12), done.
/content/pcb-defect-detection-yolo


In [3]:
!pip install -q -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 105.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 910.8/910.8 kB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 kB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 117.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires jupyter-server==2.18.2, but you have jupyter-server 2.20.0 which is incompatible.


In [4]:
from ultralytics import YOLO
import ultralytics
import torch

print("Ultralytics:", ultralytics.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics: 8.4.87
CUDA: True
GPU: Tesla T4


In [5]:
from pathlib import Path
import shutil

import cv2
import pandas as pd
from sklearn.model_selection import train_test_split

# =========================
# RUTAS
# =========================

PROJECT_ROOT = Path.cwd()

DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

DATA_RAW_DIR.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Descargar DeepPCB
!wget -q https://github.com/tangsanli5201/DeepPCB/archive/refs/heads/master.zip -O data/raw/DeepPCB-master.zip
!unzip -q data/raw/DeepPCB-master.zip -d data/raw/

DEEPPCB_DIR = DATA_RAW_DIR / "DeepPCB-master"
PCB_DATA_DIR = DEEPPCB_DIR / "PCBData"

TRAINVAL_FILE = PCB_DATA_DIR / "trainval.txt"
TEST_FILE = PCB_DATA_DIR / "test.txt"

YOLO_DATASET_DIR = DATA_PROCESSED_DIR / "deep_pcb_yolo"

assert DEEPPCB_DIR.exists(), "No se encontró DeepPCB."
assert PCB_DATA_DIR.exists(), "No se encontró PCBData."
assert TRAINVAL_FILE.exists(), "No se encontró trainval.txt."
assert TEST_FILE.exists(), "No se encontró test.txt."

print("Dataset original descargado correctamente.")


# =========================
# FUNCIONES
# =========================

def read_split_file(file_path: Path) -> list[str]:
    with open(file_path, "r", encoding="utf-8") as file:
        return [line.strip() for line in file if line.strip()]


def resolve_test_image_path(image_rel_path: str) -> Path:
    logical_path = PCB_DATA_DIR / image_rel_path

    if logical_path.exists():
        return logical_path

    return logical_path.with_name(
        logical_path.stem + "_test" + logical_path.suffix
    )


def build_samples_dataframe(samples: list[str], original_split: str) -> pd.DataFrame:
    records = []

    for sample in samples:
        image_rel_path, annotation_rel_path = sample.split()

        records.append({
            "sample_id": Path(image_rel_path).stem,
            "group": Path(image_rel_path).parts[0],
            "original_split": original_split,
            "image_path": resolve_test_image_path(image_rel_path),
            "annotation_path": PCB_DATA_DIR / annotation_rel_path,
        })

    return pd.DataFrame(records)


def validate_original_bbox(
    x_min: int,
    y_min: int,
    x_max: int,
    y_max: int,
    image_width: int,
    image_height: int,
) -> None:
    assert 0 <= x_min < x_max <= image_width
    assert 0 <= y_min < y_max <= image_height


def convert_bbox_to_yolo(
    x_min: int,
    y_min: int,
    x_max: int,
    y_max: int,
    image_width: int,
    image_height: int,
) -> tuple[float, float, float, float]:
    box_width = x_max - x_min
    box_height = y_max - y_min

    x_center = x_min + box_width / 2
    y_center = y_min + box_height / 2

    return (
        x_center / image_width,
        y_center / image_height,
        box_width / image_width,
        box_height / image_height,
    )


def convert_annotation_file_to_yolo(annotation_path: Path, image_path: Path) -> list[str]:
    image = cv2.imread(str(image_path))

    if image is None:
        raise ValueError(f"No se pudo leer la imagen: {image_path}")

    image_height, image_width = image.shape[:2]
    yolo_lines = []

    with open(annotation_path, "r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            line = line.strip()

            if not line:
                continue

            values = line.split()

            if len(values) != 5:
                raise ValueError(
                    f"Formato inválido en {annotation_path}, línea {line_number}: {line}"
                )

            x_min, y_min, x_max, y_max, original_class_id = map(int, values)

            if original_class_id not in range(1, 7):
                raise ValueError(
                    f"Clase inválida {original_class_id} en {annotation_path}, línea {line_number}"
                )

            validate_original_bbox(
                x_min=x_min,
                y_min=y_min,
                x_max=x_max,
                y_max=y_max,
                image_width=image_width,
                image_height=image_height,
            )

            x_center, y_center, box_width, box_height = convert_bbox_to_yolo(
                x_min=x_min,
                y_min=y_min,
                x_max=x_max,
                y_max=y_max,
                image_width=image_width,
                image_height=image_height,
            )

            yolo_class_id = original_class_id - 1

            yolo_lines.append(
                f"{yolo_class_id} "
                f"{x_center:.6f} "
                f"{y_center:.6f} "
                f"{box_width:.6f} "
                f"{box_height:.6f}"
            )

    return yolo_lines


def reset_yolo_dataset_directory(output_dir: Path) -> None:
    if output_dir.exists():
        shutil.rmtree(output_dir)

    for split_name in ["train", "val", "test"]:
        (output_dir / "images" / split_name).mkdir(parents=True, exist_ok=True)
        (output_dir / "labels" / split_name).mkdir(parents=True, exist_ok=True)


def process_split(split_df: pd.DataFrame, split_name: str, output_dir: Path) -> dict:
    images_output_dir = output_dir / "images" / split_name
    labels_output_dir = output_dir / "labels" / split_name

    processed_images = 0
    processed_labels = 0
    processed_annotations = 0

    for _, sample in split_df.iterrows():
        source_image_path = sample["image_path"]
        source_annotation_path = sample["annotation_path"]

        target_image_path = images_output_dir / f"{sample['sample_id']}{source_image_path.suffix}"
        target_label_path = labels_output_dir / f"{sample['sample_id']}.txt"

        shutil.copy2(source_image_path, target_image_path)

        yolo_annotations = convert_annotation_file_to_yolo(
            annotation_path=source_annotation_path,
            image_path=source_image_path,
        )

        with open(target_label_path, "w", encoding="utf-8") as file:
            file.write("\n".join(yolo_annotations))
            if yolo_annotations:
                file.write("\n")

        processed_images += 1
        processed_labels += 1
        processed_annotations += len(yolo_annotations)

    return {
        "split": split_name,
        "images": processed_images,
        "label_files": processed_labels,
        "annotations": processed_annotations,
    }


# =========================
# RECONSTRUCCIÓN DE SPLITS
# =========================

trainval_samples = read_split_file(TRAINVAL_FILE)
test_samples = read_split_file(TEST_FILE)

df_trainval = build_samples_dataframe(trainval_samples, original_split="trainval")
df_test = build_samples_dataframe(test_samples, original_split="test")

df_train, df_val = train_test_split(
    df_trainval,
    test_size=0.20,
    random_state=42,
    stratify=df_trainval["group"],
)

df_test_final = df_test.copy()

print(f"Train: {len(df_train)} imágenes")
print(f"Validation: {len(df_val)} imágenes")
print(f"Test: {len(df_test_final)} imágenes")


# =========================
# GENERACIÓN DATASET YOLO
# =========================

reset_yolo_dataset_directory(YOLO_DATASET_DIR)

processing_results = [
    process_split(df_train, "train", YOLO_DATASET_DIR),
    process_split(df_val, "val", YOLO_DATASET_DIR),
    process_split(df_test_final, "test", YOLO_DATASET_DIR),
]

df_processing_results = pd.DataFrame(processing_results)
display(df_processing_results)

assert df_processing_results["images"].sum() == 1500
assert df_processing_results["label_files"].sum() == 1500
assert df_processing_results["annotations"].sum() == 10013

print("Dataset YOLO generado correctamente.")


# =========================
# DATA.YAML
# =========================

data_yaml_content = f"""path: {YOLO_DATASET_DIR.as_posix()}
train: images/train
val: images/val
test: images/test

names:
  0: open
  1: short
  2: mousebite
  3: spur
  4: copper
  5: pin-hole
"""

data_yaml_path = YOLO_DATASET_DIR / "data.yaml"

with open(data_yaml_path, "w", encoding="utf-8") as file:
    file.write(data_yaml_content)

print(data_yaml_content)
print(f"data.yaml creado en: {data_yaml_path}")

Dataset original descargado correctamente.
Train: 800 imágenes
Validation: 200 imágenes
Test: 500 imágenes


,split,images,label_files,annotations
0,train,800,800,5513
1,val,200,200,1360
2,test,500,500,3140


Dataset YOLO generado correctamente.
path: /content/pcb-defect-detection-yolo/data/processed/deep_pcb_yolo
train: images/train
val: images/val
test: images/test

names:
  0: open
  1: short
  2: mousebite
  3: spur
  4: copper
  5: pin-hole

data.yaml creado en: /content/pcb-defect-detection-yolo/data/processed/deep_pcb_yolo/data.yaml


In [6]:
from ultralytics import YOLO
from pathlib import Path
import torch
import time

PROJECT_ROOT = Path.cwd()
RUNS_DIR = PROJECT_ROOT / "runs"
DATA_YAML_PATH = PROJECT_ROOT / "data" / "processed" / "deep_pcb_yolo" / "data.yaml"

print("CUDA disponible:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("data.yaml:", DATA_YAML_PATH)

CUDA disponible: True
GPU: Tesla T4
data.yaml: /content/pcb-defect-detection-yolo/data/processed/deep_pcb_yolo/data.yaml


In [7]:
start_time = time.time()

model_nano = YOLO("yolo11n.pt")

results_nano = model_nano.train(
    data=str(DATA_YAML_PATH),
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project=str(RUNS_DIR),
    name="yolo11n_deeppcb",
    workers=2,
    seed=42,
    patience=10,
    exist_ok=True
)

elapsed_time_nano = time.time() - start_time

print(f"Tiempo total YOLO11n: {elapsed_time_nano / 60:.2f} minutos")

Ultralytics 8.4.87 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/pcb-defect-detection-yolo/data/processed/deep_pcb_yolo/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11n_deeppcb, nbs=64, nms=False, opset=None, optimize=

In [8]:
start_time = time.time()

model_small = YOLO("yolo11s.pt")

results_small = model_small.train(
    data=str(DATA_YAML_PATH),
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project=str(RUNS_DIR),
    name="yolo11s_deeppcb",
    workers=2,
    seed=42,
    patience=10,
    exist_ok=True
)

elapsed_time_small = time.time() - start_time

print(f"Tiempo total YOLO11s: {elapsed_time_small / 60:.2f} minutos")

Ultralytics 8.4.87 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/pcb-defect-detection-yolo/data/processed/deep_pcb_yolo/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11s_deeppcb, nbs=64, nms=False, opset=None, optimize=

In [9]:
for run_name in ["yolo11n_deeppcb", "yolo11s_deeppcb"]:
    run_dir = RUNS_DIR / run_name
    print(f"\n{run_name}")
    print("Existe:", run_dir.exists())

    if run_dir.exists():
        for file in sorted(run_dir.iterdir()):
            print("-", file.name)


yolo11n_deeppcb
Existe: True
- BoxF1_curve.png
- BoxPR_curve.png
- BoxP_curve.png
- BoxR_curve.png
- args.yaml
- confusion_matrix.png
- confusion_matrix_normalized.png
- labels.jpg
- results.csv
- results.png
- train_batch0.jpg
- train_batch1.jpg
- train_batch2.jpg
- val_batch0_labels.jpg
- val_batch0_pred.jpg
- val_batch1_labels.jpg
- val_batch1_pred.jpg
- val_batch2_labels.jpg
- val_batch2_pred.jpg
- weights

yolo11s_deeppcb
Existe: True
- BoxF1_curve.png
- BoxPR_curve.png
- BoxP_curve.png
- BoxR_curve.png
- args.yaml
- confusion_matrix.png
- confusion_matrix_normalized.png
- labels.jpg
- results.csv
- results.png
- train_batch0.jpg
- train_batch1.jpg
- train_batch2.jpg
- val_batch0_labels.jpg
- val_batch0_pred.jpg
- val_batch1_labels.jpg
- val_batch1_pred.jpg
- val_batch2_labels.jpg
- val_batch2_pred.jpg
- weights


In [10]:
best_nano_path = RUNS_DIR / "yolo11n_deeppcb" / "weights" / "best.pt"
best_small_path = RUNS_DIR / "yolo11s_deeppcb" / "weights" / "best.pt"

model_nano_best = YOLO(str(best_nano_path))
model_small_best = YOLO(str(best_small_path))

test_results_nano = model_nano_best.val(
    data=str(DATA_YAML_PATH),
    split="test",
    imgsz=640,
    batch=16,
    device=0,
    project=str(RUNS_DIR),
    name="yolo11n_test",
    exist_ok=True
)

test_results_small = model_small_best.val(
    data=str(DATA_YAML_PATH),
    split="test",
    imgsz=640,
    batch=16,
    device=0,
    project=str(RUNS_DIR),
    name="yolo11s_test",
    exist_ok=True
)

Ultralytics 8.4.87 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,583,322 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 220.4±174.0 MB/s, size: 29.6 KB)
val: Scanning /content/pcb-defect-detection-yolo/data/processed/deep_pcb_yolo/labels/test... 500 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 500/500 586.0it/s 0.9s
val: New cache created: /content/pcb-defect-detection-yolo/data/processed/deep_pcb_yolo/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 32/32 5.1it/s 6.3s
                   all        500       3140      0.922      0.907      0.954      0.686
                  open        482        659      0.973      0.832      0.961      0.612
                 short        368        478      0.701      0.864      0.863      0.542
             mousebite        413        586      0.961      0.922      0.975  

In [11]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
RUNS_DIR = PROJECT_ROOT / "runs"

for path in sorted(RUNS_DIR.iterdir()):
    if path.is_dir():
        print(path)

/content/pcb-defect-detection-yolo/runs/yolo11n_deeppcb
/content/pcb-defect-detection-yolo/runs/yolo11n_test
/content/pcb-defect-detection-yolo/runs/yolo11s_deeppcb
/content/pcb-defect-detection-yolo/runs/yolo11s_test


In [12]:
import pandas as pd

run_names = [
    "yolo11n_deeppcb",
    "yolo11s_deeppcb",
]

for run_name in run_names:
    results_path = RUNS_DIR / run_name / "results.csv"
    print(f"\n{run_name}")
    print("Existe:", results_path.exists())

    if results_path.exists():
        df = pd.read_csv(results_path)
        display(df.tail())


yolo11n_deeppcb
Existe: True


,epoch,time,train/box_loss,train/cls_loss,train/dfl_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),val/box_loss,val/cls_loss,val/dfl_loss,lr/pg0,lr/pg1,lr/pg2
23,24,394.643,1.04107,0.74648,0.93654,0.95106,0.93208,0.97592,0.67546,1.02334,0.64060,0.94827,0.000545,0.000545,0.000545
24,25,409.993,1.06442,0.74020,0.94099,0.94594,0.91529,0.96545,0.58253,1.24636,0.75994,1.00838,0.000525,0.000525,0.000525
25,26,425.360,1.01663,0.71316,0.92446,0.95607,0.92331,0.97061,0.53003,1.53269,0.77748,1.09339,0.000505,0.000505,0.000505
26,27,441.086,1.02916,0.72554,0.93027,0.94354,0.92800,0.97715,0.70304,0.96645,0.63119,0.93900,0.000485,0.000485,0.000485
27,28,457.727,1.00985,0.69586,0.92567,0.96639,0.93116,0.97724,0.70093,0.99208,0.60294,0.93959,0.000465,0.000465,0.000465



yolo11s_deeppcb
Existe: True


,epoch,time,train/box_loss,train/cls_loss,train/dfl_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),val/box_loss,val/cls_loss,val/dfl_loss,lr/pg0,lr/pg1,lr/pg2
18,19,353.021,1.03544,0.60532,0.93988,0.94760,0.92080,0.96923,0.50834,1.56546,0.65127,1.10898,0.000644,0.000644,0.000644
19,20,371.767,1.09519,0.63552,0.95474,0.69859,0.67458,0.61394,0.12598,3.00867,1.08422,1.81668,0.000624,0.000624,0.000624
20,21,389.698,1.03202,0.59549,0.93335,0.82079,0.78882,0.79996,0.20631,2.68463,0.93437,1.62180,0.000604,0.000604,0.000604
21,22,408.029,1.06881,0.62275,0.94134,0.94524,0.87778,0.95014,0.50940,1.50298,0.66520,1.09238,0.000584,0.000584,0.000584
22,23,426.753,1.00172,0.57717,0.92188,0.83532,0.81822,0.83494,0.25304,2.48760,0.88514,1.49623,0.000564,0.000564,0.000564


In [13]:
test_run_names = [
    "yolo11n_test",
    "yolo11s_test",
]

for run_name in test_run_names:
    results_path = RUNS_DIR / run_name / "results.csv"
    print(f"\n{run_name}")
    print("Existe:", results_path.exists())

    if results_path.exists():
        df = pd.read_csv(results_path)
        display(df.tail())


yolo11n_test
Existe: False

yolo11s_test
Existe: False


In [14]:
for run_name in [
    "yolo11n_deeppcb",
    "yolo11s_deeppcb",
    "yolo11n_test",
    "yolo11s_test",
]:
    run_dir = RUNS_DIR / run_name

    print(f"\n{run_name}")
    if run_dir.exists():
        for file in sorted(run_dir.iterdir()):
            print("-", file.name)


yolo11n_deeppcb
- BoxF1_curve.png
- BoxPR_curve.png
- BoxP_curve.png
- BoxR_curve.png
- args.yaml
- confusion_matrix.png
- confusion_matrix_normalized.png
- labels.jpg
- results.csv
- results.png
- train_batch0.jpg
- train_batch1.jpg
- train_batch2.jpg
- val_batch0_labels.jpg
- val_batch0_pred.jpg
- val_batch1_labels.jpg
- val_batch1_pred.jpg
- val_batch2_labels.jpg
- val_batch2_pred.jpg
- weights

yolo11s_deeppcb
- BoxF1_curve.png
- BoxPR_curve.png
- BoxP_curve.png
- BoxR_curve.png
- args.yaml
- confusion_matrix.png
- confusion_matrix_normalized.png
- labels.jpg
- results.csv
- results.png
- train_batch0.jpg
- train_batch1.jpg
- train_batch2.jpg
- val_batch0_labels.jpg
- val_batch0_pred.jpg
- val_batch1_labels.jpg
- val_batch1_pred.jpg
- val_batch2_labels.jpg
- val_batch2_pred.jpg
- weights

yolo11n_test
- BoxF1_curve.png
- BoxPR_curve.png
- BoxP_curve.png
- BoxR_curve.png
- confusion_matrix.png
- confusion_matrix_normalized.png
- val_batch0_labels.jpg
- val_batch0_pred.jpg
- val_bat

In [15]:
PROJECT_ROOT = Path.cwd()
RUNS_DIR = PROJECT_ROOT / "runs"

def extract_test_metrics(results, model_name):
    """
    Extrae métricas principales de un resultado de validación YOLO.
    """
    box = results.box

    return {
        "model": model_name,
        "precision": float(box.mp),
        "recall": float(box.mr),
        "mAP50": float(box.map50),
        "mAP50_95": float(box.map),
    }


test_metrics = pd.DataFrame([
    extract_test_metrics(test_results_nano, "YOLO11n"),
    extract_test_metrics(test_results_small, "YOLO11s"),
])

test_metrics

,model,precision,recall,mAP50,mAP50_95
0,YOLO11n,0.922457,0.906759,0.953771,0.685606
1,YOLO11s,0.938966,0.914269,0.961891,0.723329


In [16]:
metrics_dir = PROJECT_ROOT / "results" / "metrics"
metrics_dir.mkdir(parents=True, exist_ok=True)

test_metrics_path = metrics_dir / "test_metrics_summary.csv"

test_metrics.to_csv(test_metrics_path, index=False)

print(f"Métricas guardadas en: {test_metrics_path}")
display(test_metrics)

Métricas guardadas en: /content/pcb-defect-detection-yolo/results/metrics/test_metrics_summary.csv


,model,precision,recall,mAP50,mAP50_95
0,YOLO11n,0.922457,0.906759,0.953771,0.685606
1,YOLO11s,0.938966,0.914269,0.961891,0.723329


In [17]:
CLASS_NAMES = [
    "open",
    "short",
    "mousebite",
    "spur",
    "copper",
    "pin-hole",
]

def extract_class_metrics(results, model_name):
    """
    Extrae métricas por clase desde un resultado de validación YOLO.
    """
    box = results.box

    rows = []

    for idx, class_name in enumerate(CLASS_NAMES):
        rows.append({
            "model": model_name,
            "class_id": idx,
            "class_name": class_name,
            "precision": float(box.p[idx]),
            "recall": float(box.r[idx]),
            "mAP50": float(box.ap50[idx]),
            "mAP50_95": float(box.ap[idx]),
        })

    return rows


class_metrics = pd.DataFrame(
    extract_class_metrics(test_results_nano, "YOLO11n")
    + extract_class_metrics(test_results_small, "YOLO11s")
)

class_metrics_path = metrics_dir / "test_metrics_by_class.csv"

class_metrics.to_csv(class_metrics_path, index=False)

display(class_metrics)
print(f"Métricas por clase guardadas en: {class_metrics_path}")

,model,class_id,class_name,precision,recall,mAP50,mAP50_95
0,YOLO11n,0,open,0.973386,0.832486,0.960576,0.611563
1,YOLO11n,1,short,0.700790,0.864017,0.863129,0.542447
2,YOLO11n,2,mousebite,0.960880,0.922128,0.975155,0.667924
3,YOLO11n,3,spur,0.962924,0.914127,0.960720,0.675406
4,YOLO11n,4,copper,0.952324,0.965517,0.978986,0.825264
5,YOLO11n,5,pin-hole,0.984440,0.942277,0.984061,0.791032
6,YOLO11s,0,open,0.938077,0.942514,0.966499,0.623456
7,YOLO11s,1,short,0.813637,0.864017,0.898452,0.594920
8,YOLO11s,2,mousebite,0.971478,0.930034,0.971534,0.699904
9,YOLO11s,3,spur,0.980151,0.886128,0.967215,0.714028


Métricas por clase guardadas en: /content/pcb-defect-detection-yolo/results/metrics/test_metrics_by_class.csv


In [18]:
!zip -r final_yolo_results.zip runs results

  adding: runs/ (stored 0%)
  adding: runs/yolo11n_deeppcb/ (stored 0%)
  adding: runs/yolo11n_deeppcb/val_batch1_pred.jpg (deflated 17%)
  adding: runs/yolo11n_deeppcb/args.yaml (deflated 54%)
  adding: runs/yolo11n_deeppcb/results.png (deflated 7%)
  adding: runs/yolo11n_deeppcb/BoxR_curve.png (deflated 9%)
  adding: runs/yolo11n_deeppcb/weights/ (stored 0%)
  adding: runs/yolo11n_deeppcb/weights/best.pt (deflated 10%)
  adding: runs/yolo11n_deeppcb/weights/last.pt (deflated 10%)
  adding: runs/yolo11n_deeppcb/train_batch2.jpg (deflated 14%)
  adding: runs/yolo11n_deeppcb/BoxF1_curve.png (deflated 7%)
  adding: runs/yolo11n_deeppcb/labels.jpg (deflated 30%)
  adding: runs/yolo11n_deeppcb/val_batch0_labels.jpg (deflated 17%)
  adding: runs/yolo11n_deeppcb/val_batch1_labels.jpg (deflated 19%)
  adding: runs/yolo11n_deeppcb/val_batch0_pred.jpg (deflated 15%)
  adding: runs/yolo11n_deeppcb/confusion_matrix_normalized.png (deflated 20%)
  adding: runs/yolo11n_deeppcb/BoxP_curve.png (defla

In [19]:
from google.colab import files
files.download("final_yolo_results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>